In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import pandas as pd
from sklearn.metrics import classification_report

print("CUDA disponible:", torch.cuda.is_available())
print("Versión CUDA:", torch.version.cuda)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


c:\Users\cance\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA disponible: False
Versión CUDA: None


device(type='cpu')

In [12]:
MODEL_NAME = "CohereLabs/aya-23-8B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 4/4 [00:06<00:00,  1.66s/it]
Some parameters are on the meta device because they were offloaded to the cpu.


In [3]:
df = pd.read_csv("../data/dataset_nueva_limpio.zip", compression="zip")

# Nos quedamos con la parte en español y las etiquetas
df = df[["texto_limpio", "label"]].dropna(subset=["texto_limpio"]).reset_index(drop=True)
df["label"] = df["label"].astype(int)

df.head(), df["label"].value_counts()


(                                        texto_limpio  label
 0  Aquí está el falso negro, Shaun King, pensando...      0
 1  Una defensora de Tennessee Trump está extasiad...      0
 2  Únete a Patrick cada semana aquí en 21WIRE.TV ...      0
 3  Esto es lo que sucede cuando los racistas no c...      0
 4  El presidente estadounidense Donald Trump firm...      1,
 label
 1    20825
 0    16542
 Name: count, dtype: int64)

In [ ]:
import re

def clasificar_aya(texto: str) -> int:
    """
    Devuelve 0 si Aya cree que la noticia es FALSA,
    1 si cree que es VERDADERA.
    """

    prompt = (
    "Clasifica la siguiente noticia como 0=falsa o 1=verdadera.\n"
    "Reglas:\n"
    "- Marca 1 (verdadera) si el texto parece una noticia periodística normal: coherente, descriptiva, "
    "con hechos plausibles, instituciones reales, lugares, fechas o citas.\n"
    "- Marca 0 (falsa) solo si hay señales claras de desinformación: insultos, lenguaje muy emocional o agresivo, "
    "teorías conspirativas, exageraciones extremas, estructura caótica o texto claramente propagandístico.\n"
    "- Si el texto es neutral, informativo o no muestra señales fuertes de falsedad, clasifícalo como 1.\n"
    "- No seas demasiado estricto: recuerda que muchas noticias son verdaderas aunque no tengas toda la evidencia.\n"
    "Responde SOLO con 0 o 1.\n\n"
    "Ejemplos:\n"
    "FALSO -> 0: 'Aquí está el falso negro... Sanders... #DemDebate'\n"
    "FALSO -> 0: 'Trump llama títere, desastre, WEAK on Crime... Rusia Rusia Rusia...'\n"
    "VERDADERO -> 1: 'Singapur planea desplegar autobuses autónomos en 2022 para mejorar el transporte público.'\n"
    "VERDADERO -> 1: 'El Servicio Secreto investiga un incidente con un fotógrafo de Time en un mitin de Trump.'\n\n"
    "Noticia:\n"
    f"{texto}\n\n"
    "Respuesta:"
    )


    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=4,
            do_sample=False,      # greedy
            temperature=0.0,      # determinista
        )

    # Tomamos solo lo generado (sin el prompt)
    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    gen_text = tokenizer.decode(generated, skip_special_tokens=True)
    gen_text = gen_text.strip()

    # Buscamos 0 o 1 explícito
    match = re.search(r"[01]", gen_text)
    if match:
        return int(match.group(0))
    else:
        # Si no encontramos nada claro, por defecto 0 (falsa)
        return 0


In [ ]:
from tqdm import tqdm
import pandas as pd
from sklearn.metrics import classification_report

# 1) CREAR SUBSET BALANCEADO DE 5000 NOTICIAS
df_falsas = df[df["label"] == 0].sample(2500, random_state=42)
df_reales = df[df["label"] == 1].sample(2500, random_state=42)

df_eval = pd.concat([df_falsas, df_reales]).sample(frac=1, random_state=42).reset_index(drop=True)

print("Total noticias:", len(df_eval))
print(df_eval["label"].value_counts())

# 2) CLASIFICAR CON AYA (CON BARRA DE PROGRESO)
preds = []

for texto in tqdm(df_eval["texto_limpio"], total=len(df_eval), desc="Clasificando con Aya23", unit="noticia"):
    pred = clasificar_aya(texto)
    preds.append(pred)

df_eval["pred"] = preds

# 3) REPORTE FINAL
print("\n=== RESULTADOS ===")
print(classification_report(df_eval["label"], df_eval["pred"]))


Total noticias: 5000
label
0    2500
1    2500
Name: count, dtype: int64


Clasificando con Aya23:  39%|███▊      | 1932/5000 [6:31:49<5:13:09,  6.12s/noticia]    

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import json

y_true = df_eval["label"]
y_pred = df_eval["pred"]

acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec  = recall_score(y_true, y_pred)
f1   = f1_score(y_true, y_pred)

# Matriz de confusión: [[TN, FP],
#                       [FN, TP]]
cm = confusion_matrix(y_true, y_pred)

# Crear diccionario metrics
metrics = {
    "accuracy": acc,
    "precision": prec,
    "recall": rec,
    "f1": f1,
    "confusion_matrix": cm.tolist()  # convertir a lista para guardar en JSON
}

# Guardar JSON
with open("../Resultados/metricas_aya23.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

# Mostrar métricas
print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", rec)
print("F1:", f1)

print("\nMatriz de confusión:")
print(cm)